# MobileNetV2 Stage 3 — D3 External Inference Sustainability (Frozen Classification Checkpoints)

**Model:** standalone MobileNetV2, not the MobileNetV2 constituent from REMNET. **Checkpoint source:** verified Stage 2 classification `.pt` files (`MobileNetV2_timm_seed*.pt`). **Dataset:** 3,497 fixed D3 RealWaste external samples; no retraining, fine-tuning, class remapping, or threshold adjustment.

This notebook follows the supplied `mobilenet_v2_co2(2).ipynb` for historical sustainability records and the matching frozen MobileNetV2 Stage 3 D3 classification notebook for checkpoint restoration. Checkpoints are loaded only if model-state keys and tensor shapes match exactly; `strict=True` is always required.

**Stage 2 historical training** and **Stage 3 D3 external inference** are reported separately. CodeCarbon's *original* `emissions.csv` values are preserved; standardized emissions use `energy_kWh × 0.475 kg CO₂e/kWh` in separate columns. All formatted output uses fixed decimals. Inference is monitored after checkpoint loading and warm-up. The same `384 × 384` RGB normalization, batch 32, 15-second CodeCarbon interval, minimum two full passes, and five seeds apply to the Stage 3 D3 benchmarking protocol.

## 1. Fixed configuration and input files

In [2]:
from __future__ import annotations
import os, sys, json, hashlib, zipfile, subprocess, importlib.util, traceback, re
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone

# Kaggle input location confirmed by the user; adjust ONLY if Kaggle moves the dataset.
INPUT_ROOT = Path('/kaggle/input/datasets/akash264208/manifist-part2/Section3')
OUTPUT = Path('/kaggle/working/MobileNetV2_D3_Inference_Sustainability')
OUTPUT.mkdir(parents=True, exist_ok=True)
SEEDS = (7, 49, 112, 345, 476)
MODEL_NAME = 'MobileNetV2'
CARBON_FACTOR = 0.475  # standardized kg CO2e / kWh (keep CodeCarbon original separately)
MIN_COMPLETE_PASSES = 2
MIN_MONITOR_SECONDS = 35.0
CODECARBON_SAMPLING_SECONDS = 15
ZIP_PATH = Path('/kaggle/working/MobileNetV2_D3_Inference_Sustainability_All_Outputs.zip')
# MobileNetV2 constructor is verified against actual checkpoint tensor names/shapes.
CLASSES = ('cardboard','glass','metal','paper','plastic','trash')
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}
EXPECTED_EXTERNAL_COUNT = 3497
EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT = '0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0'
# Set this to the exact Kaggle folder containing the D3 Section 2 CSV files.
# Example: '/kaggle/input/datasets/YOUR_ACCOUNT/YOUR_D3_SECTION2_OUTPUTS'
# If None, the notebook locates unique matching files under /kaggle/input.
D3_MANIFEST_ROOT = '/kaggle/input/datasets/akash264208/manifist-part2/Section3/manifist'
BATCH_SIZE = 32

# Optional genuine historical STANDALONE MobileNetV2 prediction files.
HISTORICAL_REFERENCE_DIR = None
REQUIRE_HISTORICAL_REFERENCE = False  # Set True to STOP external evaluation if references are absent.

assert INPUT_ROOT.is_dir(), f'Input dataset was not attached: {INPUT_ROOT}'
print('Kaggle input:', INPUT_ROOT)
print('Output directory:', OUTPUT)
print('Seeds:', SEEDS)
print('Frozen standalone MobileNetV2 classification checkpoints, Stage 3 D3 sustainability; no training or fine-tuning.')

Kaggle input: /kaggle/input/datasets/akash264208/manifist-part2/Section3
Output directory: /kaggle/working/MobileNetV2_D3_Inference_Sustainability
Seeds: (7, 49, 112, 345, 476)
Frozen standalone MobileNetV2 classification checkpoints, Stage 3 D3 sustainability; no training or fine-tuning.


## 2. Dependencies and strict standalone MobileNetV2 classification-checkpoint inventory

In [3]:
# Exact CodeCarbon version and monitoring implementation used in companion Stage 3 notebooks.
try:
    from importlib.metadata import version as installed_version
    _cc_version=installed_version('codecarbon')
except Exception:
    _cc_version=None
if _cc_version != '3.3.2':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','codecarbon==3.3.2'])
from codecarbon import EmissionsTracker
import time, shutil, platform
print('CodeCarbon version:',installed_version('codecarbon'))

for name in ('kagglehub', 'timm'):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', name])
import kagglehub, timm, re, numpy as np, pandas as pd, torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cuda':
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False
print('Device:',DEVICE,'| torch:',torch.__version__,'| timm:',timm.__version__)

def only_directory_with_name(root,name):
    found=sorted(p for p in root.rglob('*') if p.is_dir() and p.name.casefold()==name.casefold())
    if len(found)!=1:raise RuntimeError(f'Expected exactly one directory {name} under {root}; found {found}')
    return found[0]

def unique_by_filename(root,filename):
    found=sorted(p for p in root.rglob(filename) if p.is_file())
    if len(found)!=1:raise RuntimeError(f'Expected one {filename} under {root}: {found}')
    return found[0]

CKPT_ROOT=only_directory_with_name(INPUT_ROOT,'Classification_checkpoints')
REMNET_CKPT_DIR=only_directory_with_name(CKPT_ROOT,'Remnet')
RESULTS_ROOT=only_directory_with_name(INPUT_ROOT,'per_seed_performance_summary')

def d3_csv_path(filename):
    root = Path(D3_MANIFEST_ROOT) if D3_MANIFEST_ROOT else Path('/kaggle/input')
    if not root.is_dir():
        raise FileNotFoundError(f'D3 manifest folder not attached: {root}')
    return unique_by_filename(root, filename)

EXTERNAL_MANIFEST = d3_csv_path('D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv')
CLASS_MAPPING = d3_csv_path('D3_Section2_Class_Mapping.csv')

# Select ONLY the standalone MobileNetV2 directory, never REMNET ensemble checkpoints.
def normalized_name(name):
    return re.sub(r'[^a-z0-9]','',name.lower())
standalone_dirs=[p for p in CKPT_ROOT.iterdir() if p.is_dir()
                 and 'mobilenetv2' in normalized_name(p.name)
                 and 'remnet' not in normalized_name(p.name)]
if len(standalone_dirs)!=1:
    raise RuntimeError('Expected exactly one standalone MobileNetV2 directory under '
                       f'{CKPT_ROOT}; found {standalone_dirs}. Inspect Kaggle attachments.')
STANDALONE_CKPT_DIR=standalone_dirs[0]
print('Standalone MobileNetV2 checkpoint directory:',STANDALONE_CKPT_DIR)

CHECKPOINTS={}
for seed in SEEDS:
    matches=sorted(p for p in STANDALONE_CKPT_DIR.rglob('*.pt') if
                   re.search(rf'(?:^|[^0-9])seed[_-]?{seed}(?:[^0-9]|$)',p.stem,re.IGNORECASE))
    if len(matches)!=1:
        raise RuntimeError(f'Expected precisely one standalone MobileNetV2 .pt checkpoint for seed {seed}: {matches}')
    CHECKPOINTS[seed]=matches[0]

all_pts = sorted(CKPT_ROOT.rglob('*.pt'))
standalone_pts=sorted(STANDALONE_CKPT_DIR.rglob('*.pt'))
remnet_pts = sorted(REMNET_CKPT_DIR.rglob('*.pt'))
assert len(standalone_pts) == 5, f'Expected 5 standalone MobileNetV2 checkpoints: {standalone_pts}'
assert len(remnet_pts) == 15, f'Expected 15 REMNET ensemble constituent checkpoints, found {len(remnet_pts)}'
assert len(all_pts) == 30, f'Expected 30 total Stage 2 checkpoints, found {len(all_pts)}'
assert not set(CHECKPOINTS.values()).intersection(remnet_pts)
for seed,path in CHECKPOINTS.items():print('Standalone MobileNetV2',seed,':',path)
for path in [EXTERNAL_MANIFEST, CLASS_MAPPING]: print('CSV:', path)

# Historical performance metrics are not used for checkpoint selection.
HISTORICAL_PER_SEED_FILES=sorted(p for p in RESULTS_ROOT.rglob('*.csv') if 'mobilenet' in p.name.lower() and 'per_seed' in p.name.lower())
HISTORICAL_SUMMARY_FILES=sorted(p for p in RESULTS_ROOT.rglob('*.csv') if 'mobilenet' in p.name.lower() and ('summary' in p.name.lower() or '5_seed' in p.name.lower()))
HISTORICAL_PER_SEED=HISTORICAL_PER_SEED_FILES[0] if len(HISTORICAL_PER_SEED_FILES)==1 else None
HISTORICAL_SUMMARY=HISTORICAL_SUMMARY_FILES[0] if len(HISTORICAL_SUMMARY_FILES)==1 else None

CodeCarbon version: 3.3.2
Device: cuda | torch: 2.11.0+cu130 | timm: 1.0.30
Standalone MobileNetV2 checkpoint directory: /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/mobilenet_v2
Standalone MobileNetV2 7 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/mobilenet_v2/MobileNetV2_timm_seed7.pt
Standalone MobileNetV2 49 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/mobilenet_v2/MobileNetV2_timm_seed49.pt
Standalone MobileNetV2 112 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/mobilenet_v2/MobileNetV2_timm_seed112.pt
Standalone MobileNetV2 345 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/mobilenet_v2/MobileNetV2_timm_seed345.pt
Standalone MobileNetV2 476 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/mobilenet_v2/MobileNetV2_timm_seed476.pt
CSV: /

## 3. Validate the locked D3 manifest and six-class RealWaste mapping

In [4]:
def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for b in iter(lambda:stream.read(chunk_size),b''):
            h.update(b)
    return h.hexdigest()

def canonical_external_fingerprint(df):
    h = hashlib.sha256()
    for row in df.sort_values('external_relative_path').itertuples(index=False):
        h.update((row.external_relative_path+'|'+str(int(row.output_index))+'|'+
                  row.sha256+'\n').encode('utf-8'))
    return h.hexdigest()

external_df = pd.read_csv(EXTERNAL_MANIFEST, dtype=str, keep_default_na=False)
mapping_df = pd.read_csv(CLASS_MAPPING, dtype=str, keep_default_na=False)

assert set(['external_relative_path','class','output_index','sha256','final_status']) <= set(external_df.columns)
assert set(['d3_source_label','remnet_trained_class','remnet_output_index','include_for_primary_d3']) <= set(mapping_df.columns)
assert len(external_df)==EXPECTED_EXTERNAL_COUNT, 'Unexpected D3 manifest count (should be 3497).'
assert external_df['sha256'].nunique()==EXPECTED_EXTERNAL_COUNT, 'Duplicate hashes in external manifest.'
assert external_df['external_relative_path'].nunique()==EXPECTED_EXTERNAL_COUNT
assert set(external_df['class']) == set(CLASSES)
assert external_df['final_status'].eq('ELIGIBLE_AFTER_REFERENCE_VERIFICATION').all(), 'Noneligible images in frozen manifest.'
included_mapping = mapping_df.loc[mapping_df['include_for_primary_d3'].str.lower().eq('true')].copy()
assert len(included_mapping) == 6
assert set(included_mapping['remnet_trained_class']) == set(CLASSES)
for item in included_mapping.itertuples(index=False):
    assert int(item.remnet_output_index) == CLASS_TO_ID[item.remnet_trained_class]
    if 'class_review' in mapping_df.columns:
        assert item.class_review == 'CONFIRMED_BY_USER'
assert set(external_df['source_category']) == set(included_mapping['d3_source_label'])
assert all(int(i)==CLASS_TO_ID[c] for c,i in zip(external_df['class'],external_df['output_index']))

EXTERNAL_FINGERPRINT=canonical_external_fingerprint(external_df)
assert EXTERNAL_FINGERPRINT == EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT, (EXTERNAL_FINGERPRINT,'External cohort fingerprint mismatch')
print('D3 frozen manifest verified:',len(external_df),'Fingerprint:',EXTERNAL_FINGERPRINT)
print('Six class mapping:',CLASS_TO_ID)
print('External per-class counts:',external_df['class'].value_counts().sort_index().to_dict())

D3 frozen manifest verified: 3497 Fingerprint: 0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0
Six class mapping: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
External per-class counts: {'cardboard': 454, 'glass': 419, 'metal': 757, 'paper': 500, 'plastic': 889, 'trash': 478}


## 4. Load the attached RealWaste D3 dataset

In [5]:
# Stage 3 D3 dataset path — fixed mounted Kaggle input.
EXTERNAL_HANDLE = 'akash264208/real-waste'
external_download = Path('/kaggle/input/datasets/akash264208/real-waste')
external_download_method = 'attached Kaggle dataset'
assert external_download.is_dir(), f'D3 RealWaste dataset missing: {external_download}'
print('D3 RealWaste:', external_download)

D3 RealWaste: /kaggle/input/datasets/akash264208/real-waste


## 5. Resolve D3 images and verify every locked image hash

In [6]:
IMG_EXT={'.jpg','.jpeg','.png','.webp','.bmp'}

def safe_parts(relative):
    p=PurePosixPath(str(relative).replace('\\','/'))
    if p.is_absolute() or not p.parts or any(s in ('','.','..') for s in p.parts):
        raise ValueError(f'Unsafe image relative path: {relative!r}')
    return p.parts

def external_class_roots(root):
    expected=set(included_mapping['d3_source_label'])
    candidates=[root] + sorted((p for p in root.rglob('*') if p.is_dir()), key=str)
    return [p for p in candidates if all((p/c).is_dir() for c in expected)]

def resolve_external(external,download):
    candidates=external_class_roots(download)
    print('D3 RealWaste class-root candidates:',candidates)
    if not candidates: raise FileNotFoundError('Missing RealWaste source-category directories.')
    expected_rows=[(safe_parts(r.external_relative_path),r.sha256,r.source_category)
                   for _,r in external.iterrows()]
    for root in candidates:
        try:
            dirs=[p for p in root.iterdir() if p.is_dir()]
            image_counts={p.name:sum(x.is_file() and x.suffix.lower() in IMG_EXT
                         for x in p.rglob('*')) for p in dirs}
            if len(image_counts)!=9 or sum(image_counts.values())!=4752:
                raise RuntimeError(f'Expected nine RealWaste folders / 4752 images: {image_counts}')
            if sum(image_counts[c] for c in included_mapping['d3_source_label'])!=3587:
                raise RuntimeError('D3 six-category source total differs from Section 2.')
            resolved=[]
            for parts,digest,source_cls in expected_rows:
                image=root.joinpath(*parts)
                if not image.is_file() or parts[0]!=source_cls:
                    raise FileNotFoundError(str(image))
                if file_sha256(image)!=digest:
                    raise ValueError(f'D3 image hash mismatch: {image}')
                resolved.append(str(image))
            print('D3 RealWaste images SHA-256 verified:',len(resolved),'root:',root)
            return resolved,root,image_counts
        except (FileNotFoundError,ValueError,RuntimeError) as exc:
            print('D3 candidate rejected:',root,'reason:',str(exc)[:180])
    raise RuntimeError('No RealWaste root matches the 3497 locked image hashes.')

external_paths, external_image_root, d3_counts=resolve_external(external_df,external_download)
external_df=external_df.copy(); external_df['resolved_path']=external_paths
external_eval_df=external_df.sort_values('external_relative_path').reset_index(drop=True)
assert len(external_eval_df)==3497
print('External evaluated images:',len(external_eval_df))

D3 RealWaste class-root candidates: [PosixPath('/kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste')]
D3 RealWaste images SHA-256 verified: 3497 root: /kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste
External evaluated images: 3497


## 6. Restore strict standalone MobileNetV2 Stage 2 checkpoints and preprocessing

Determine the exact matching torchvision or timm MobileNetV2 constructor by verifying **all** parameter names and shapes. Never bypass incompatible weights, use `strict=False`, or silently replace classification weights.

In [7]:
TEST_TRANSFORM=transforms.Compose([
    transforms.Resize((384,384)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])

# Identify the REAL checkpoint constructor by exact weight-key AND shape matching.
# Do not use strict=False or manually reshape/reinitialize any checkpoint weights.
class MobileNetV2_Model(nn.Module):
    """Original torchvision implementation, retained as a compatible candidate."""
    def __init__(self,num_classes):
        super().__init__()
        self.model=models.mobilenet_v2(weights=None)
        self.model.classifier[1]=nn.Linear(self.model.classifier[1].in_features,num_classes)
    def forward(self,x):
        return self.model(x)

TIMM_MOBILENET_ARCH = 'auto'
TIMM_ARCH_CANDIDATES = (
    'mobilenetv2_100','mobilenetv2_050','mobilenetv2_075',
    'mobilenetv2_110d','mobilenetv2_120d','mobilenetv2_140',
    'tf_mobilenetv2_100','tf_mobilenetv2_075','tf_mobilenetv2_050',
)

def _weight_keys_and_shapes_match(candidate_state, checkpoint_state):
    return (set(candidate_state)==set(checkpoint_state) and all(
        tuple(candidate_state[key].shape)==tuple(checkpoint_state[key].shape)
        for key in candidate_state))

def _candidate_states(state):
    """Allow only recognizable wrapper prefixes, preserving all tensor values."""
    variants={'unaltered':state}
    for prefix in ('module.','_orig_mod.','model.','net.'):
        if all(k.startswith(prefix) for k in state):
            variants['remove_'+prefix] = {k[len(prefix):]:v for k,v in state.items()}
    variants['add_model.']={'model.'+k:v for k,v in state.items()}
    if all(k.startswith('module.') for k in state):
        stripped={k[7:]:v for k,v in state.items()}
        variants['remove_module_add_model.']={'model.'+k:v for k,v in stripped.items()}
    if all(k.startswith('model.module.') for k in state):
        variants['remove_inner_module.']={'model.'+k[len('model.module.'):]:v for k,v in state.items()}
    return variants

def _construct_matching_mobilenet(state,checkpoint_path):
    alternatives=_candidate_states(state)
    matches=[]
    torchvision_model=MobileNetV2_Model(6)
    for conversion,converted in alternatives.items():
        if _weight_keys_and_shapes_match(torchvision_model.state_dict(),converted):
            matches.append(('torchvision','torchvision.mobilenet_v2',conversion,torchvision_model,converted))
    if matches:
        return matches[0]
    del torchvision_model

    options = (TIMM_ARCH_CANDIDATES if TIMM_MOBILENET_ARCH=='auto' else (TIMM_MOBILENET_ARCH,))
    valid=set(timm.list_models('*mobilenetv2*'))
    for name in options:
        if name not in valid:
            continue
        model=timm.create_model(name,pretrained=False,num_classes=6)
        expected=model.state_dict()
        for conversion,converted in alternatives.items():
            if _weight_keys_and_shapes_match(expected,converted):
                matches.append(('timm',name,conversion,model,converted))
                break
        if not any(m[1]==name for m in matches):
            del model
    if not matches:
        example_keys=list(state.keys())[:8]
        raise RuntimeError(
            f'No exact torchvision/timm MobileNetV2 state_dict match for {checkpoint_path.name}. '
            f'Checkpoint contains {len(state)} keys, starting with {example_keys}. '
            'Please verify the checkpoint architecture/model registry. '
            'No weights were skipped or changed.')
    if len(matches)>1:
        raise RuntimeError(
            'Multiple exact timm architecture matches for checkpoint '+checkpoint_path.name+': '+
            str([f'{x[1]} ({x[2]})' for x in matches])+'. '
            'Set TIMM_MOBILENET_ARCH to the verified Stage 2 architecture.')
    return matches[0]

class LockedImageDataset(Dataset):
    def __init__(self,df,kind):
        self.df=df.reset_index(drop=True)
        self.kind=kind
        self.path_col='external_relative_path'
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        with Image.open(row['resolved_path']) as im:
            x=TEST_TRANSFORM(im.convert('RGB'))
        return x,int(CLASS_TO_ID[row['class']]),i

def locked_loader(df,kind):
    return DataLoader(LockedImageDataset(df,kind),batch_size=BATCH_SIZE,
                      shuffle=False,num_workers=0,pin_memory=DEVICE.type=='cuda')

def restore_checkpoint(seed):
    p=CHECKPOINTS[seed]  # STRICTLY the standalone MobileNetV2 checkpoint folder.
    ckpt=torch.load(p,map_location='cpu',weights_only=True)
    if not isinstance(ckpt,dict):
        raise RuntimeError(f'Unexpected checkpoint container: {p}')
    state_key=next((k for k in ('state_dict','model_state_dict') if k in ckpt),None)
    if state_key is not None:
        state=ckpt[state_key]
        if 'seed' in ckpt and int(ckpt['seed'])!=seed:
            raise RuntimeError(f'Checkpoint seed mismatch: {p}')
        if 'model' in ckpt and 'mobilenet' not in re.sub(r'[^a-z0-9]','',str(ckpt['model']).lower()):
            raise RuntimeError(f'Checkpoint model mismatch: {p}: {ckpt["model"]}')
        if 'num_classes' in ckpt and int(ckpt['num_classes'])!=6:
            raise RuntimeError(f'Expected six checkpoint classes: {p}')
    else:
        state=ckpt
    if not isinstance(state,dict) or not state or not all(torch.is_tensor(v) for v in state.values()):
        raise RuntimeError(f'Checkpoint does not expose a tensor-only state_dict: {p}')
    framework,arch,conversion,model,adjusted=_construct_matching_mobilenet(state,p)
    model.load_state_dict(adjusted,strict=True)
    model.to(DEVICE).eval()
    for param in model.parameters():param.requires_grad_(False)
    if not hasattr(restore_checkpoint,'detected_architectures'):
        restore_checkpoint.detected_architectures={}
    restore_checkpoint.detected_architectures[seed]={
        'framework':framework,'architecture':arch,'prefix_conversion':conversion}
    print(f'Validated standalone seed {seed}: {framework}/{arch} '
          f'(prefix mapping: {conversion}; strict=True)')
    return model,ckpt,file_sha256(p)

def infer(model,df,kind):
    labels=[]; probs=[]; positions=[]
    with torch.inference_mode():
        for images,y,index in locked_loader(df,kind):
            pred=torch.softmax(model(images.to(DEVICE,non_blocking=True)),dim=1)
            probs.append(pred.cpu().numpy().astype(np.float64))
            labels.append(y.numpy());positions.append(index.numpy())
    labels=np.concatenate(labels).astype(np.int64)
    probs=np.concatenate(probs)
    positions=np.concatenate(positions)
    assert np.array_equal(positions,np.arange(len(df))), 'Inference image order changed'
    assert np.array_equal(labels,df['class'].map(CLASS_TO_ID).to_numpy()), 'Class-index mismatch'
    assert probs.shape==(len(df),6) and np.isfinite(probs).all(), 'Invalid probability array'
    assert np.allclose(probs.sum(axis=1),1.0,atol=2e-6), 'Probabilities do not sum to 1'
    return labels,probs

_smoke,_smoke_info,_smoke_digest=restore_checkpoint(SEEDS[0])
with torch.inference_mode():
    _test_logits=_smoke(torch.zeros(1,3,384,384,device=DEVICE))
assert tuple(_test_logits.shape)==(1,6)
print('MobileNetV2 strict-load and six-class smoke test: PASS')
del _smoke,_smoke_info,_test_logits
if DEVICE.type=='cuda':torch.cuda.empty_cache()

Validated standalone seed 7: timm/mobilenetv2_100 (prefix mapping: unaltered; strict=True)
MobileNetV2 strict-load and six-class smoke test: PASS


## 7. CodeCarbon inference measurement boundaries

Measure **D3 inference only**. Restore frozen classification weights and complete two warm-up batches **before** starting CodeCarbon. Inside monitoring, process all 3,497 D3 images in full passes, with at least 2 complete passes and a minimum 35-second interval. Synchronize CUDA for timing, then stop tracking. Batch loading and preprocessing are included in the monitored interval; checkpoint loading and warm-up are excluded. Preserve original CodeCarbon emissions and independently compute emissions using the declared 0.475 factor.

In [8]:
LOG_ROOT = OUTPUT/'CodeCarbon_Logs'
LOG_ROOT.mkdir(parents=True,exist_ok=True)

def make_tracker(log_dir, seed):
    log_dir.mkdir(parents=True,exist_ok=True)
    prior = log_dir/'emissions.csv'
    if prior.exists():
        prior.rename(log_dir/f'emissions_previous_{time.time_ns()}.csv')
    options = dict(
        project_name=f'MobileNetV2_Stage3_seed{seed}_External3497',
        output_dir=str(log_dir), output_file='emissions.csv',
        save_to_file=True, tracking_mode='machine',
        measure_power_secs=CODECARBON_SAMPLING_SECONDS,
        pue=1.0, log_level='error'
    )
    if DEVICE.type=='cuda': options['gpu_ids']=[0]
    return EmissionsTracker(**options)

def gpu_sync():
    if DEVICE.type=='cuda': torch.cuda.synchronize()

def measured_row(log_dir):
    filepath=log_dir/'emissions.csv'
    if not filepath.is_file():
        raise FileNotFoundError(f'CodeCarbon emissions file missing: {filepath}')
    df=pd.read_csv(filepath)
    if df.empty: raise RuntimeError(f'Empty CodeCarbon file: {filepath}')
    row=df.iloc[-1].to_dict()
    energy=float(row['energy_consumed'])
    if not np.isfinite(energy) or energy<0: raise RuntimeError(f'Invalid CodeCarbon energy {energy}')
    emissions=float(row['emissions']) if 'emissions' in row and pd.notna(row['emissions']) else np.nan
    return row,energy,emissions

def monitored_mobilenetv2_inference(seed, model):
    loader=locked_loader(external_eval_df,'external')
    model.eval()
    # Two identical warm-up forward-only batches, before monitoring starts.
    images, _, _ = next(iter(loader))
    images=images.to(DEVICE,non_blocking=True)
    with torch.inference_mode():
        for _ in range(2):
            p=torch.softmax(model(images),dim=1)
            if p.shape[1]!=6: raise RuntimeError('Unexpected six-class output during warm-up')
    gpu_sync()
    del images,p
    log_dir=LOG_ROOT/f'MobileNetV2_seed{seed}'/'inference'
    tracker=make_tracker(log_dir,seed)
    passes=0
    processed=0
    labels_first=[]; pred_first=[]
    started=False
    try:
        tracker.start()
        started=True
        gpu_sync()
        t0=time.perf_counter()
        with torch.inference_mode():
            while passes<MIN_COMPLETE_PASSES or (time.perf_counter()-t0)<MIN_MONITOR_SECONDS:
                current_positions=[]
                for images,y,positions in loader:
                    images=images.to(DEVICE,non_blocking=True)
                    probs=torch.softmax(model(images),dim=1)
                    if tuple(probs.shape)!=(len(y),6):
                        raise RuntimeError('Six-class output mismatch')
                    if passes==0:
                        labels_first.extend(y.tolist())
                        pred_first.extend(probs.argmax(dim=1).cpu().tolist())
                        current_positions.extend(positions.tolist())
                    processed+=len(y)
                if passes==0 and current_positions!=list(range(len(external_eval_df))):
                    raise RuntimeError('External image order changed during monitored inference')
                passes+=1
        gpu_sync()
        seconds=time.perf_counter()-t0
    finally:
        if started:
            gpu_sync()
            tracker.stop()
    if processed!=passes*len(external_eval_df):
        raise RuntimeError(f'Incomplete inference pass: {processed} images / {passes} passes')
    row,energy,cc_emissions=measured_row(log_dir)
    if seconds<=0 or processed<=0:raise RuntimeError('Invalid measured workload')
    ytrue=np.asarray(labels_first,dtype=int)
    ypred=np.asarray(pred_first,dtype=int)
    expected=external_eval_df['class'].map(CLASS_TO_ID).to_numpy()
    assert np.array_equal(ytrue,expected)
    accuracy_pct=100*float((ytrue==ypred).mean())
    return {
       'model':'MobileNetV2','seed':int(seed),
       'unique_external_images':len(external_eval_df),
       'number_complete_repetitions':passes,'total_processed_images':processed,
       'batch_size':BATCH_SIZE,'input_resolution':'384x384',
       'monitoring_seconds':seconds,'total_inference_energy_kWh':energy,
       'codecarbon_emissions_kgCO2e':cc_emissions,
       'standardized_CO2e_kg_fixed_factor':energy*CARBON_FACTOR,
       'standardized_CO2_factor_kg_per_kWh':CARBON_FACTOR,
       'energy_kWh_per_processed_image':energy/processed,
       'time_seconds_per_processed_image':seconds/processed,
       'throughput_images_per_second':processed/seconds,
       'first_pass_accuracy_pct_diagnostic':accuracy_pct,
       'codecarbon_duration_seconds':row.get('duration',np.nan),
       'codecarbon_energy_column':'energy_consumed',
       'codecarbon_version':installed_version('codecarbon'),
       'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU',
       'raw_emissions_csv':str(log_dir/'emissions.csv')
    }


## 8. Run five frozen standalone MobileNetV2 checkpoints on external D3

Record original checkpoint-file SHA-256, verified exact model architecture, class mapping, 3,497 D3 images, full repetitions, runtime, CodeCarbon energy, original emissions and standardized CO₂e. Never use REMNET constituent checkpoint weights.

In [9]:
sustainability_rows=[]
checkpoint_rows=[]
for seed in SEEDS:
    print(f'\n=== STANDALONE MOBILENETV2 STAGE 3 D3 INFERENCE — SEED {seed} ===',flush=True)
    model,meta,digest=restore_checkpoint(seed)
    architecture_info=restore_checkpoint.detected_architectures.get(seed,{})
    checkpoint_rows.append({
        'model':'MobileNetV2','seed':seed,'checkpoint_path':str(CHECKPOINTS[seed]),
        'checkpoint_file_sha256':digest,
        'checkpoint_seed':meta.get('seed') if isinstance(meta, dict) else None,
        'checkpoint_model':meta.get('model') if isinstance(meta, dict) else None,
        'checkpoint_architecture_metadata':meta.get('architecture') if isinstance(meta, dict) else None,
        'strictly_matched_framework':architecture_info.get('framework'),
        'strictly_matched_architecture':architecture_info.get('architecture'),
        'verified_prefix_conversion':architecture_info.get('prefix_conversion'),
        'split_sha256':meta.get('split_sha256') if isinstance(meta, dict) else None,
        'classification_checkpoint_source':'standalone Classification_checkpoints/mobilenet_v2/*.pt',
        'preprocessing':'RGB Resize(384,384) ToTensor Normalize(.5,.5,.5)'})
    result=monitored_mobilenetv2_inference(seed,model)
    sustainability_rows.append(result)
    pd.DataFrame(sustainability_rows).to_csv(OUTPUT/'MobileNetV2_Stage3_D3_Inference_Per_Seed.csv',index=False)
    pd.DataFrame(checkpoint_rows).to_csv(OUTPUT/'MobileNetV2_Stage3_D3_Classification_Checkpoint_Registry.csv',index=False)
    print(f"Seed {seed} completed | D3 external: {result['unique_external_images']} unique images × "
          f"{result['number_complete_repetitions']} complete passes = {result['total_processed_images']} processed "
          f"| energy {result['total_inference_energy_kWh']:.9f} kWh "
          f"| original CodeCarbon CO2e {result['codecarbon_emissions_kgCO2e']:.9f} kg "
          f"| standardized CO2e {result['standardized_CO2e_kg_fixed_factor']:.9f} kg "
          f"| monitored runtime {result['monitoring_seconds']:.3f} s",flush=True)
    del model
    if DEVICE.type=='cuda':torch.cuda.empty_cache()

checkpoint_df=pd.DataFrame(checkpoint_rows)
checkpoint_df.to_csv(OUTPUT/'MobileNetV2_Stage3_D3_Classification_Checkpoint_Registry.csv',index=False)



=== STANDALONE MOBILENETV2 STAGE 3 D3 INFERENCE — SEED 7 ===
Validated standalone seed 7: timm/mobilenetv2_100 (prefix mapping: unaltered; strict=True)


[codecarbon WARNING @ 13:33:23] Multiple instances of codecarbon are allowed to run at the same time.


Seed 7 completed | D3 external: 3497 unique images × 2 complete passes = 6994 processed | energy 0.001356479 kWh | original CodeCarbon CO2e 0.000224558 kg | standardized CO2e 0.000644328 kg | monitored runtime 76.622 s

=== STANDALONE MOBILENETV2 STAGE 3 D3 INFERENCE — SEED 49 ===
Validated standalone seed 49: timm/mobilenetv2_100 (prefix mapping: unaltered; strict=True)
Seed 49 completed | D3 external: 3497 unique images × 2 complete passes = 6994 processed | energy 0.001364436 kWh | original CodeCarbon CO2e 0.000225875 kg | standardized CO2e 0.000648107 kg | monitored runtime 74.136 s

=== STANDALONE MOBILENETV2 STAGE 3 D3 INFERENCE — SEED 112 ===
Validated standalone seed 112: timm/mobilenetv2_100 (prefix mapping: unaltered; strict=True)
Seed 112 completed | D3 external: 3497 unique images × 2 complete passes = 6994 processed | energy 0.001390894 kWh | original CodeCarbon CO2e 0.000230255 kg | standardized CO2e 0.000660675 kg | monitored runtime 74.248 s

=== STANDALONE MOBILENETV2 

## 9. Inference results: mean ± sample SD across five seeds (`ddof=1`), fixed-decimal output

In [10]:
results_df=pd.DataFrame(sustainability_rows)
assert len(results_df)==5 and set(results_df.seed)==set(SEEDS)
MEASURES=[
    'total_inference_energy_kWh','codecarbon_emissions_kgCO2e',
    'standardized_CO2e_kg_fixed_factor','monitoring_seconds',
    'energy_kWh_per_processed_image','time_seconds_per_processed_image',
    'throughput_images_per_second','first_pass_accuracy_pct_diagnostic']
summary_rows=[]
for measure in MEASURES:
    values=results_df[measure].astype(float)
    summary_rows.append({'model':'MobileNetV2','measure':measure,'n_seeds':len(values),
                         'mean':values.mean(),'sample_sd':values.std(ddof=1),
                         'min':values.min(),'max':values.max()})
summary_df=pd.DataFrame(summary_rows)
results_df.to_csv(OUTPUT/'MobileNetV2_Stage3_D3_Inference_Per_Seed.csv',index=False,float_format='%.12f')
summary_df.to_csv(OUTPUT/'MobileNetV2_Stage3_D3_Inference_Five_Seed_Mean_SD.csv',index=False,float_format='%.12f')
print(summary_df[['measure','mean','sample_sd']].to_string(index=False,
    formatters={'mean':lambda v:f'{v:.12f}','sample_sd':lambda v:f'{v:.12f}'}))
print('Five-seed D3 inference summary saved.')


                           measure            mean      sample_sd
        total_inference_energy_kWh  0.001384064906 0.000023557370
       codecarbon_emissions_kgCO2e  0.000229124239 0.000003899792
 standardized_CO2e_kg_fixed_factor  0.000657430830 0.000011189751
                monitoring_seconds 74.515027407200 1.196255835093
    energy_kWh_per_processed_image  0.000000197893 0.000000003368
  time_seconds_per_processed_image  0.010654136032 0.000171040297
      throughput_images_per_second 93.879240566371 1.477745387512
first_pass_accuracy_pct_diagnostic 40.943665999428 2.745388898958
Five-seed D3 inference summary saved.


## 10. Historical Stage 2 standalone MobileNetV2 training — reference only

Values below are transcribed from the **actual saved five-seed output** of the uploaded `mobilenet_v2_co2(2).ipynb`. Its five-seed means contain higher precision than the displayed per-seed table. Preserve those original means exactly; sample standard deviations from the displayed seed table are approximate because those rows were printed with six decimal places. Stage 2 original CodeCarbon emissions are **not** 0.475-standardized. Separately compute a 0.475 standardized CO₂e estimate from the printed training-energy figures. These training results are historical and should never be represented as newly measured Stage 3 energy.

In [11]:
# Historical MobileNetV2 training values from actual saved outputs in mobilenet_v2_co2(2).ipynb.
# These are STAGE 2 training records, not inference energy and NOT new Stage 3 experiments.
historical_rows=[
    {'seed':7,  'stage2_codecarbon_CO2e_kg_reported':0.002176,'training_energy_kWh_reported':0.013147,'training_time_min_reported':9.057387},
    {'seed':49, 'stage2_codecarbon_CO2e_kg_reported':0.002211,'training_energy_kWh_reported':0.013353,'training_time_min_reported':9.118122},
    {'seed':112,'stage2_codecarbon_CO2e_kg_reported':0.002211,'training_energy_kWh_reported':0.013358,'training_time_min_reported':9.132188},
    {'seed':345,'stage2_codecarbon_CO2e_kg_reported':0.002211,'training_energy_kWh_reported':0.013356,'training_time_min_reported':9.131784},
    {'seed':476,'stage2_codecarbon_CO2e_kg_reported':0.002212,'training_energy_kWh_reported':0.013362,'training_time_min_reported':9.133216},
]
historical_training_df=pd.DataFrame(historical_rows)
historical_training_df['stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy']=(
    historical_training_df['training_energy_kWh_reported']*CARBON_FACTOR)
historical_training_df['stage2_original_source']='mobilenet_v2_co2(2).ipynb, saved output (rounded printed per-seed table)'
historical_training_df['status']='Historical Stage 2 standalone MobileNetV2 training; not remeasured in Stage 3'
historical_training_df.to_csv(OUTPUT/'MobileNetV2_Stage2_Historical_Training_Per_Seed.csv',
                              index=False,float_format='%.12f')
reported_historical_means={
    'stage2_codecarbon_CO2e_kg_reported':0.00220424386779135,
    'training_energy_kWh_reported':0.01331511932717624,
    'training_time_min_reported':9.114539311726888,
}
historical_summary=[]
for measure in ['stage2_codecarbon_CO2e_kg_reported','training_energy_kWh_reported',
                'training_time_min_reported','stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy']:
    vals=historical_training_df[measure].astype(float)
    historical_summary.append({
        'measure':measure,'n_seeds':5,
        'mean_as_reported_stage2_or_from_rounded_data':reported_historical_means.get(measure,float(vals.mean())),
        'mean_from_printed_rounded_seed_rows':float(vals.mean()),
        'sample_sd_from_printed_rounded_seed_rows':float(vals.std(ddof=1)),
        'note':'Historical original means from saved Stage 2 notebook; SDs and standardized emissions calculated from rounded printed seed rows'})
historical_summary_df=pd.DataFrame(historical_summary)
historical_summary_df.to_csv(OUTPUT/'MobileNetV2_Stage2_Historical_Training_Mean_SD.csv',
                             index=False,float_format='%.12f')

print('\n'+'='*75)
print('HISTORICAL STAGE 2 — STANDALONE MOBILENETV2 TRAINING (NOT MEASURED IN STAGE 3)')
print('='*75)
print(historical_training_df[['seed','stage2_codecarbon_CO2e_kg_reported',
       'training_energy_kWh_reported','stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy',
       'training_time_min_reported']].to_string(index=False,formatters={
       'stage2_codecarbon_CO2e_kg_reported':lambda v:f'{v:.9f}',
       'training_energy_kWh_reported':lambda v:f'{v:.9f}',
       'stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy':lambda v:f'{v:.9f}',
       'training_time_min_reported':lambda v:f'{v:.6f}',
    }))
print('Historical Stage 2 original CodeCarbon CO2e mean (kg):',f"{reported_historical_means['stage2_codecarbon_CO2e_kg_reported']:.12f}")
print('Historical Stage 2 training energy mean (kWh):',f"{reported_historical_means['training_energy_kWh_reported']:.12f}")
print('Historical Stage 2 training time mean (min):',f"{reported_historical_means['training_time_min_reported']:.9f}")
print('Historical Stage 2 standardized CO2e from rounded seed energies (kg):',
      f"{historical_training_df['stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy'].mean():.12f}")
print('Stage 2 historical data is separate from Stage 3 D3 measured inference; no combined-cost claim.')

print('\n'+'='*75)
print('STAGE 3 — D3 EXTERNAL MOBILENETV2 INFERENCE (NEWLY MONITORED; 3,497 UNIQUE IMAGES)')
print('='*75)
print(results_df[['seed','unique_external_images','number_complete_repetitions','total_processed_images',
    'total_inference_energy_kWh','codecarbon_emissions_kgCO2e',
    'standardized_CO2e_kg_fixed_factor','monitoring_seconds',
    'energy_kWh_per_processed_image','time_seconds_per_processed_image',
    'throughput_images_per_second']].to_string(index=False,formatters={
       'total_inference_energy_kWh':lambda v:f'{v:.9f}',
       'codecarbon_emissions_kgCO2e':lambda v:f'{v:.9f}',
       'standardized_CO2e_kg_fixed_factor':lambda v:f'{v:.9f}',
       'monitoring_seconds':lambda v:f'{v:.3f}',
       'energy_kWh_per_processed_image':lambda v:f'{v:.12f}',
       'time_seconds_per_processed_image':lambda v:f'{v:.9f}',
       'throughput_images_per_second':lambda v:f'{v:.3f}',
    }))
print('Original CodeCarbon emissions and standardized emissions at 0.475 kg CO2e/kWh remain separate.')
print('Five-seed mean standardized CO2e (kg):',
      f"{results_df['standardized_CO2e_kg_fixed_factor'].mean():.9f}")
print('Five-seed sample SD standardized CO2e (kg):',
      f"{results_df['standardized_CO2e_kg_fixed_factor'].std(ddof=1):.9f}")



HISTORICAL STAGE 2 — STANDALONE MOBILENETV2 TRAINING (NOT MEASURED IN STAGE 3)
 seed stage2_codecarbon_CO2e_kg_reported training_energy_kWh_reported stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy training_time_min_reported
    7                        0.002176000                  0.013147000                                                0.006244825                   9.057387
   49                        0.002211000                  0.013353000                                                0.006342675                   9.118122
  112                        0.002211000                  0.013358000                                                0.006345050                   9.132188
  345                        0.002211000                  0.013356000                                                0.006344100                   9.131784
  476                        0.002212000                  0.013362000                                                0.006346950            

## 11. Provenance configuration, full raw logs, and one downloadable ZIP

In [12]:
configuration={
 'experiment':'Standalone MobileNetV2 Stage 3 D3 external inference sustainability',
 'stage2_sustainability_source':'mobilenet_v2_co2(2).ipynb, historical saved notebook outputs',
 'historical_stage2_training':'Reference only; not remeasured or combined with D3 external inference',
 'historical_stage2_original_CO2':'Original CodeCarbon results reported in Stage 2 notebook',
 'historical_stage2_standardized_CO2':'Rounded historical energy times 0.475; distinct from original emissions',
 'checkpoint_source':'Frozen Stage 2 standalone MobileNetV2 classification checkpoints',
 'input_checkpoint_root':str(STANDALONE_CKPT_DIR),
 'not_ensemble_constituent_checkpoints':True,
 'seeds':list(SEEDS),'class_to_index':CLASS_TO_ID,
 'external_dataset':EXTERNAL_HANDLE,
 'd3_manifest_csv_path':str(EXTERNAL_MANIFEST),
 'external_unique_images':len(external_eval_df),
 'external_manifest_fingerprint':EXTERNAL_FINGERPRINT,
 'd1_original_test_evaluation_performed':False,
 'preprocessing':'RGB Resize((384,384)) ToTensor Normalize((0.5,)*3,(0.5,)*3)',
 'architecture':'Verified exact standalone MobileNetV2 torchvision/timm architecture and strict checkpoint state',
 'no_training_no_adaptation':True,'classification_checkpoints_unmodified':True,
 'checkpoint_architecture_by_seed':getattr(restore_checkpoint,'detected_architectures',{}),
 'checkpoint_loading':'Exact key-and-shape architecture matching; strict=True; seed metadata checks',
 'batch_size':BATCH_SIZE,'minimum_complete_passes':MIN_COMPLETE_PASSES,
 'minimum_monitor_seconds':MIN_MONITOR_SECONDS,
 'codecarbon_sampling_seconds':CODECARBON_SAMPLING_SECONDS,
 'codecarbon_version':installed_version('codecarbon'),
 'device':str(DEVICE),
 'hardware_gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
 'carbon_factor_fixed_kgCO2e_per_kWh':CARBON_FACTOR,
 'measurement_boundary':'After checkpoint loading and warm-up; full D3 passes, including batch loading/preprocessing; GPU synchronized for timing; no checkpoint load/warm-up in monitored interval.',
 'original_codecarbon_emissions':'Original emissions.csv emissions column, never replaced',
 'standardized_CO2e':'Energy in kWh times fixed carbon-intensity factor; separately displayed',
 'external_3497_test_sha256_verified':True,
 'fair_comparison_note':'Use identical hardware, batch 32, preprocessing and monitoring protocol; confirm classification checkpoint SHA256 registry matches frozen MobileNetV2 D3 classification run.',
 'created_utc':datetime.now(timezone.utc).isoformat()
}
(OUTPUT/'MobileNetV2_Stage3_D3_Inference_Configuration.json').write_text(
    json.dumps(configuration,indent=2,ensure_ascii=False),encoding='utf-8')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file() and 'original_source_extracted' not in p.parts:
            z.write(p,p.relative_to(OUTPUT))
print('One downloadable archive:',ZIP_PATH)
print('Archive bytes:',ZIP_PATH.stat().st_size)
print('Outputs: standalone MobileNetV2 Stage 3 D3 inference five-seed CSVs, original CodeCarbon logs, checkpoint registry, historical standalone MobileNetV2 Stage 2 tables, configuration JSON.')


One downloadable archive: /kaggle/working/MobileNetV2_D3_Inference_Sustainability_All_Outputs.zip
Archive bytes: 9057
Outputs: standalone MobileNetV2 Stage 3 D3 inference five-seed CSVs, original CodeCarbon logs, checkpoint registry, historical standalone MobileNetV2 Stage 2 tables, configuration JSON.
